In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from matplotlib.animation import FuncAnimation,PillowWriter
from scipy.cluster.hierarchy import dendrogram
from scipy.cluster.hierarchy import fcluster



In [ ]:
file_path=r"E:\Python\data\uav_trajectories.csv"
dataset=pd.read_csv(file_path,dtype={'timestamp': float})
pd.set_option('display.float_format', '{:.5f}'.format)
dataset=dataset.interpolate(method='linear')
dataset=dataset.drop_duplicates(subset=['timestamp'], keep='last')
dataset

,timestamp,tx,ty,tz
0,1700250023.49152,-20.11050,2.79242,17.07152
1,1700250023.59152,-20.02775,2.78101,17.10497
2,1700250023.69152,-19.95896,2.77923,17.14850
3,1700250023.79152,-19.89828,2.80470,17.22112
4,1700250023.89152,-19.85786,2.86869,17.27456
...,...,...,...,...
766135,1699986237.17464,-31.41072,-2.44510,5.09962
766136,1699986237.27464,-31.69249,-2.68590,5.08811
766137,1699986237.37464,-31.96578,-2.90151,5.08130
766138,1699986237.47464,-32.26979,-3.12411,5.08085


In [ ]:
#定义绘制轨迹动画的函数
def ani(dataset,num_poses=510):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    line,=ax.plot([],[],[],markersize=4,color='blue',label='Trajectory')
    ax.set_title("3D_Animation")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    def init():
        line.set_data([],[])
        line.set_3d_properties([])
        return line
    def update(frame):
        line.set_data(dataset["tx"][:frame],dataset["ty"][:frame])
        line.set_3d_properties(dataset["tz"][:frame])
        ax.set_xlim(min(dataset["tx"])-1,max(dataset["tx"])+1)
        ax.set_ylim(min(dataset["ty"])-1,max(dataset["ty"])+1)
        ax.set_zlim(min(dataset["tz"])-1,max(dataset["tz"])+1)
        return line
    ani=FuncAnimation(fig,update,frames=num_poses,init_func=init,blit=False)
    ani.save(r'E:\bishe\work\results\animation.gif',writer=PillowWriter(fps=30))
    plt.show()
    
            

In [ ]:
#对时间序列分割，划分出单独的无人机轨迹
def split_trajectory_segments(data,dt_thresh=0.2,dish_thresh=2):
    segments=[]
    current_seg=[data[0]]
    for i in range(1,len(data)):
        prev=data[i-1]
        curr=data[i]
        dt=curr[0]-prev[0]
        dx=curr[1]-prev[1]
        dy=curr[2]-prev[2]
        dz=curr[3]-prev[3]
        dish=np.sqrt(dx**2+dy**2+dz**2)
        if dt>dt_thresh or dish>dish_thresh:
            segments.append(np.array(current_seg))
            current_seg=[curr]
        else:
            current_seg.append(curr)
    if current_seg:
        segments.append(np.array(current_seg))
    return segments

tra_seg=split_trajectory_segments(dataset.values)
len(tra_seg)

5093

In [ ]:
for i in range(len(tra_seg)):
    time = tra_seg[i][:,0]
    dt = np.array(pd.DataFrame(time).diff())
    if dt[2] < 0.0999999 or dt[0] > 1.0000001:
        print(f"Segment {i} has irregular time intervals.")

Segment 20 has irregular time intervals.
Segment 21 has irregular time intervals.
Segment 30 has irregular time intervals.
Segment 46 has irregular time intervals.
Segment 50 has irregular time intervals.
Segment 53 has irregular time intervals.
Segment 56 has irregular time intervals.
Segment 73 has irregular time intervals.
Segment 82 has irregular time intervals.
Segment 96 has irregular time intervals.
Segment 100 has irregular time intervals.
Segment 101 has irregular time intervals.
Segment 102 has irregular time intervals.
Segment 111 has irregular time intervals.
Segment 119 has irregular time intervals.
Segment 135 has irregular time intervals.
Segment 137 has irregular time intervals.
Segment 149 has irregular time intervals.
Segment 150 has irregular time intervals.
Segment 153 has irregular time intervals.
Segment 155 has irregular time intervals.
Segment 163 has irregular time intervals.
Segment 170 has irregular time intervals.
Segment 182 has irregular time intervals.
Se

In [ ]:
print(tra_seg[0])

[[ 1.70025002e+09 -2.01104984e+01  2.79242420e+00  1.70715160e+01]
 [ 1.70025002e+09 -2.00277470e+01  2.78100726e+00  1.71049703e+01]
 [ 1.70025002e+09 -1.99589619e+01  2.77922945e+00  1.71484974e+01]
 [ 1.70025002e+09 -1.98982822e+01  2.80470295e+00  1.72211193e+01]
 [ 1.70025002e+09 -1.98578558e+01  2.86868790e+00  1.72745626e+01]
 [ 1.70025002e+09 -1.98425208e+01  2.97994539e+00  1.73078023e+01]
 [ 1.70025002e+09 -1.98568961e+01  3.11234415e+00  1.73222773e+01]
 [ 1.70025002e+09 -1.98991462e+01  3.27251786e+00  1.73291080e+01]
 [ 1.70025002e+09 -1.99553370e+01  3.41013274e+00  1.73218378e+01]
 [ 1.70025002e+09 -2.00363765e+01  3.55049845e+00  1.73172773e+01]
 [ 1.70025002e+09 -2.01251820e+01  3.67311462e+00  1.73184327e+01]
 [ 1.70025002e+09 -2.02292723e+01  3.79316300e+00  1.73226028e+01]
 [ 1.70025002e+09 -2.03280187e+01  3.88947175e+00  1.73265697e+01]
 [ 1.70025002e+09 -2.04337515e+01  3.98315235e+00  1.73263108e+01]
 [ 1.70025002e+09 -2.05276622e+01  4.04966807e+00  1.73230409e

In [ ]:
# def remove_outliers(df, z_thresh=3):
#     count = 0
#     len_list = []
#     dataset = []
#     for i in range(len(df)):
#         data = df[i]
#         data_clean = data.copy()
#         prev_len = len(data_clean)
#         for col in [1,2,3]:
#             mean = data_clean[:,col].mean()
#             std = data_clean[:,col].std()
#             z_scores = (data_clean[:,col] - mean) / std
#             data_clean = data_clean[np.abs(z_scores) < z_thresh]
#         out_len = len(data_clean)
#         if out_len != prev_len:
#             count+=1
#         len_list.append(prev_len)
#         dataset.append(data_clean)
#     return dataset,count,len_list
# data_clean,count,len_list = remove_outliers(tra_seg)
# print(f"Removed {count} outliers.")

In [ ]:
def compute_velocity(position_sequence):
    diff = np.diff(position_sequence, axis=0)
    dt = diff[:, 0] 
    velocity = diff[:, 1:] / dt[:, np.newaxis]# 利用广播机制保持维度清晰
    bearing = np.arctan2(velocity[:,1], velocity[:,0]) * 180 / np.pi
    return velocity,(bearing + 360) % 360

In [ ]:
from scipy.interpolate import make_interp_spline

posearray=[]
for i, seg in enumerate(tra_seg):
    spl_x = make_interp_spline(seg[:,0],seg[:,1],k=3)
    spl_y = make_interp_spline(seg[:,0],seg[:,2],k=3)
    spl_z = make_interp_spline(seg[:,0],seg[:,3],k=3)
    new_t = np.linspace(seg[0,0], seg[-1,0], num=len(seg))
    x_smooth = spl_x(new_t)
    y_smooth = spl_y(new_t) 
    z_smooth = spl_z(new_t)
    seg = np.stack([new_t, x_smooth, y_smooth, z_smooth], axis=1)
    velocity, bearing = compute_velocity(seg)
    vel_pad = np.concatenate([np.zeros((1, velocity.shape[1])), velocity], axis=0)
    bearing = bearing.reshape(-1, 1)
    bearing_pad = np.concatenate([bearing[0:1], bearing], axis=0)
    pose = np.concatenate([seg, vel_pad], axis=1)
    posearray.append(pose)
    
posearray[4].shape


(146, 7)

In [ ]:
print(posearray[4][:5])

[[ 1.70026692e+09 -1.41504774e+01 -2.80294323e+01  5.77124119e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 1.70026692e+09 -1.43675943e+01 -2.82831575e+01  5.76984309e+00
  -2.17117130e+00 -2.53725468e+00 -1.39809848e-02]
 [ 1.70026692e+09 -1.45342542e+01 -2.84745922e+01  5.76660883e+00
  -1.66660039e+00 -1.91434897e+00 -3.23426173e-02]
 [ 1.70026692e+09 -1.46808668e+01 -2.86395456e+01  5.75741462e+00
  -1.46612717e+00 -1.64953476e+00 -9.19421990e-02]
 [ 1.70026692e+09 -1.47860304e+01 -2.87597685e+01  5.75713984e+00
  -1.05163697e+00 -1.20223051e+00 -2.74780629e-03]]


In [ ]:
for i in range(len(posearray)):
    time = posearray[i][:,0]
    dt = np.array(pd.DataFrame(time).diff())
    if np.min(dt[2]) < 0.0999999 or np.max(dt[0]) > 1.0000001:
        print(f"Segment {i} has irregular time intervals.")

In [ ]:
# with open(r"E:\bishe\work\results\posearray.pkl","wb")as f:
#     pickle.dump(posearray,f)

In [ ]:
#统计划分出的轨迹段数量，形状，并进行z分数标准化
# from scipy import stats
# max_len=0
# max_idx=0
# tra_seg_norm=[]
# for i in range(0,len(tra_seg)):
#     if len(tra_seg[i])>max_len:
#         max_idx=i
#         max_len=len(tra_seg[i])
#     tra_seg[i]=pd.DataFrame(tra_seg[i],columns=['time','tx','ty','tz'])
#     tra_seg_norm.append(stats.zscore(tra_seg[i]))
#     # print(f'第{i}组轨迹数据形状:', tra_seg[i].shape)
    
# print(f'最长的轨迹序列为第{max_idx}组，长度为：',max_len)

In [ ]:
#计算5093个多维时间序列的dtw距离矩阵
# sequences = []
# for seg_df in tra_seg_norm:
#     seg_3d = seg_df[['tx', 'ty', 'tz']].values.astype(np.float64)
#     sequences.append(seg_3d)

# dist_matrix = dtw_ndim.distance_matrix_fast(
#     sequences ,
#     parallel=True,   
#     use_c=True,      
#     window=5      
# )
# pd.DataFrame(dist_matrix).to_csv(r"C:\Users\34362\Desktop\LSTM\results\dist_matrix.csv")
# dist_matrix

In [ ]:
# SciPy linkage聚类，以dtw计算距离
# model3 = clustering.LinkageTree(dtw_ndim.distance_matrix_fast, {})
# cluster_idx = model3.fit(sequences)

# plt.figure(figsize=(15, 8))
# dendrogram(cluster_idx)
# plt.title("DTW linkage Clustering")
# plt.xlabel("Sequence Index")
# plt.ylabel("Distance")
# plt.savefig(r"C:\Users\34362\Desktop\LSTM\results\DTW linkage Clustering.png")  # 保存
# plt.show()

# print("已保存DTW linkage Clustering.png")

In [ ]:
#获取分类结果（label为1，轨迹为圆形。label为2，轨迹为8字形）
# labels = fcluster(cluster_idx, t=2, criterion='maxclust')
# labels=pd.DataFrame(labels)
# print(labels.head(20))

In [ ]:
#用label对序列进行分类
# def classify(datas,labels):
#     datas_labels=[]
#     circular=[]
#     infinity_like=[]
#     for i in range(0,len(datas)):
#         datas_labels.append([labels.values[i],tra_seg[i]])
    
#     for i in range(0,len(datas_labels)):
#         if datas_labels[i][0]==1:
#             circular.append(datas_labels[i][1])
#         else:
#             infinity_like.append(datas_labels[i][1])
#     return circular,infinity_like

# circular,infinity_like=classify(tra_seg,labels)

    

In [ ]:
#随机绘图
# a=np.random.randint(0,len(circular),dtype=int)
# b=np.random.randint(0,len(infinity_like),dtype=int)
# ani(pd.DataFrame(circular[a]))
# ani(pd.DataFrame(infinity_like[b]))

In [ ]:
# with open(r"C:\Users\34362\Desktop\LSTM\results\circular.pkl","wb")as f:
#     pickle.dump(circular,f)
# with open(r"C:\Users\34362\Desktop\LSTM\results\infinity_like.pkl","wb")as f:
#     pickle.dump(infinity_like,f)